# GCN 训练（PyTorch Geometric · Notebook 版）

本 notebook 与 `train_pyg.py` 逻辑等价，用 PyG 的 **`GCNConv`** 替代手写图卷积层。
它与 `train.ipynb` 共用**同一套数据与划分**，因此两个 notebook 的精度可以直接比较。

**预期结果**（seed=123）：Cora **79.50%**（手写版 76.70%，论文 81.5%）。

> 为什么不用 `torch_geometric.datasets.Planetoid`？它会联网下载并自己切分数据，
> 一旦划分不同，精度差异就分不清是「实现差异」还是「划分差异」。

## 0 环境检查

In [1]:
import os, sys, time
from pathlib import Path

# 这个 notebook 放在 gcn-pytorch/notebooks/ 下，代码包在上一级。
# 两种运行方式都兼容：Jupyter 里 cwd 通常是 notebook 所在目录；
# 用 nbclient 批量执行时 cwd 可能是项目根。
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
sys.path.insert(0, str(ROOT))

import numpy as np
import torch

# 固定种子：notebook 里改超参重跑时，结果仍然可复现
SEED = 123
np.random.seed(SEED)
torch.manual_seed(SEED)

print('torch  :', torch.__version__)
print('numpy  :', np.__version__)
print('工作目录:', ROOT.name)

# 项目内的包（sys.path 已加好，可直接 import）
from gcn import silence_sparse_warning          # 稀疏张量告警上下文
from gcn.utils import (load_data,                # 数据加载（pickle / csv）
                       preprocess_adj,           # Ã = A + I 后对称归一化
                       chebyshev_polynomials)    # K 阶切比雪夫基
from gcn.models import GCN, MLP                  # 手写实现的两个模型


try:
    import torch_geometric
    from torch_geometric.nn import GCNConv
except ImportError:
    raise SystemExit('缺少 torch_geometric，请先执行: pip install torch_geometric')

print('pyg    :', torch_geometric.__version__,
      '（纯 Python 包，不需要 torch-scatter）')

from IPython.display import HTML, display


def svg_lines(series, title='', w=780, h=230, y_label=''):
    """把多条折线画成内联 SVG（不依赖 matplotlib）。

    :param series: {名称: 数字列表}，所有曲线共用同一条 x 轴（epoch）
    """
    pad_l, pad_r, pad_t, pad_b = 56, 96, 30, 34
    pw, ph = w - pad_l - pad_r, h - pad_t - pad_b
    n = max(len(v) for v in series.values())
    lo = min(min(v) for v in series.values())
    hi = max(max(v) for v in series.values())
    span = (hi - lo) or 1.0
    # 上下各留 8% 余量，曲线不贴边
    lo, hi = lo - span * 0.08, hi + span * 0.08
    if min(min(v) for v in series.values()) >= 0:
        lo = max(0.0, lo)          # 损失/准确率非负，轴不要出现负数刻度
    span = hi - lo

    colors = ['#e45756', '#4c78a8', '#54a24b', '#b279a2', '#f58518']
    parts = ['<svg width="%d" height="%d" viewBox="0 0 %d %d" '
             'xmlns="http://www.w3.org/2000/svg" style="font:12px sans-serif">'
             % (w, h, w, h)]
    parts.append('<text x="%d" y="18" font-size="13" font-weight="600" '
                 'fill="#333">%s</text>' % (pad_l, title))

    # 网格线与 y 轴刻度（4 条）
    for k in range(5):
        y = pad_t + ph * k / 4
        val = hi - span * k / 4
        parts.append('<line x1="%d" y1="%.1f" x2="%d" y2="%.1f" '
                     'stroke="#eee"/>' % (pad_l, y, pad_l + pw, y))
        parts.append('<text x="%d" y="%.1f" text-anchor="end" fill="#888">'
                     '%.3f</text>' % (pad_l - 6, y + 4, val))

    for i, (name, vals) in enumerate(series.items()):
        pts = []
        for j, v in enumerate(vals):
            x = pad_l + (pw * j / max(1, n - 1))
            y = pad_t + ph * (1 - (v - lo) / span)
            pts.append('%.1f,%.1f' % (x, y))
        c = colors[i % len(colors)]
        parts.append('<polyline points="%s" fill="none" stroke="%s" '
                     'stroke-width="1.6"/>' % (' '.join(pts), c))
        ly = pad_t + 14 + i * 18
        parts.append('<rect x="%d" y="%d" width="10" height="3" fill="%s"/>'
                     % (pad_l + pw + 12, ly - 3, c))
        parts.append('<text x="%d" y="%d" fill="#444">%s</text>'
                     % (pad_l + pw + 26, ly, name))

    parts.append('<text x="%d" y="%d" fill="#888">epoch</text>'
                 % (pad_l + pw // 2, h - 10))
    parts.append('<text x="12" y="%d" fill="#888">%s</text>'
                 % (pad_t + ph // 2, y_label))
    parts.append('</svg>')
    return HTML(''.join(parts))


def html_table(headers, rows, caption=''):
    """把二维数据渲染成简洁的 HTML 表格。"""
    th = ''.join('<th style="text-align:left;padding:4px 12px;'
                 'border-bottom:2px solid #ddd">{0}</th>'.format(h)
                 for h in headers)
    body = ''
    for r in rows:
        # 注意用 format 而不是 %：单元格里可能是 tuple（如张量形状），
        # 用 % 会被当成多个占位参数而报错
        tds = ''.join('<td style="padding:4px 12px;border-bottom:1px solid #eee">'
                      '{0}</td>'.format(c) for c in r)
        body += '<tr>{0}</tr>'.format(tds)
    cap = ('<div style="font-weight:600;margin:6px 0">{0}</div>'.format(caption)
           if caption else '')
    return HTML('<div style="font:13px sans-serif">{0}<table>{1}{2}'
                '</table></div>'.format(cap, '<tr>{0}</tr>'.format(th), body))

torch  : 2.14.1+cpu
numpy  : 2.5.4
工作目录: gcn-pytorch


D:\VMwareShareFolder\ccnu\Fusion\GNN\scripts\gcn-pytorch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pyg    : 2.8.1 （纯 Python 包，不需要 torch-scatter）


D:\VMwareShareFolder\ccnu\Fusion\GNN\scripts\gcn-pytorch\.venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 1 数据：转成 PyG 的 `Data`

`to_pyg_data` 内部仍调用 `load_data`，只是把结果重排成 PyG 能吃的形状：

| 字段 | 形状 | 说明 |
|---|---|---|
| `x` | (N,F) 稠密 float32 | 行归一化后的特征（PyG 要求稠密） |
| `edge_index` | (2,2E) int64 | 双向各一条，**不含自环** |
| `y` | (N,) int64 | 类别下标 |
| `train/val/test_mask` | (N,) bool | 划分 |

**`edge_index` 为什么不含自环**：`GCNConv` 默认 `add_self_loops=True` 会自己加。
若这里再加一遍，等价于 `A + 2I`，权重失真——这是最常见的错误。

In [2]:
from pyg_gcn import to_pyg_data, describe

DATASET = 'cora'        # 'citeseer' / 'pubmed'
SOURCE = 'pickle'       # 'csv' 亦可

data = to_pyg_data(DATASET, source=SOURCE)
print(describe(data))

src, dst = data.edge_index
fwd = set(zip(src.tolist(), dst.tolist()))
print('edge_index 对称 :', all((b, a) in fwd for a, b in fwd))
print('edge_index 自环 :', int((src == dst).sum()),
      '（应为 0，交给 GCNConv 添加）')
print('x 是否稠密      :', not data.x.is_sparse, '| dtype', data.x.dtype)
print('y  范围         :', int(data.y.min()), '~', int(data.y.max()))

nodes=2708 features=1433 edges=10556(directed) classes=7 train/val/test=140/500/1000
edge_index 对称 : True
edge_index 自环 : 0 （应为 0，交给 GCNConv 添加）
x 是否稠密      : True | dtype torch.float32
y  范围         : 0 ~ 6


## 2 模型

`PyGGCN` = 若干 `GCNConv` 堆叠。几个和手写版对齐的细节：

- `bias=False`：论文两层均无偏置。
- **不使用 `GCNConv(weight_decay=...)`**：那个参数对**每一层**的原始梯度加 L2，
  而论文只对**第一层**做正则。为与手写版可比，损失里手动加，且只加第一层。
- `num_layers` 可调，这是 PyG 版相对手写版的便利之处（手写版固定两层）。

In [3]:
from pyg_gcn.models import PyGGCN, count_parameters

HIDDEN, LAYERS, DROPOUT, LR, WD = 16, 2, 0.5, 0.01, 5e-4
EPOCHS, PATIENCE = 200, 10

torch.manual_seed(SEED)
model = PyGGCN(input_dim=data.x.shape[1], output_dim=data.num_classes,
               hidden1=HIDDEN, num_layers=LAYERS,
               dropout=DROPOUT, weight_decay=WD)

print('层数           :', LAYERS)
print('可训练参数     :', f'{count_parameters(model):,}')
print('理论值 F×H+H×C :', f'{data.x.shape[1] * HIDDEN + HIDDEN * data.num_classes:,}')
for i, conv in enumerate(model.convs):
    print(f'  convs[{i}] GCNConv{tuple(conv.lin.weight.shape)}  bias={conv.bias is not None}')

层数           : 2
可训练参数     : 23,040
理论值 F×H+H×C : 23,040
  convs[0] GCNConv(16, 1433)  bias=False
  convs[1] GCNConv(7, 16)  bias=False


## 3 训练

优化器、学习率、早停判据与 `train.py` 完全一致，因此两个 notebook 的差异
只剩下「图卷积层的实现」和「归一化的自环约定」。

In [4]:
import torch.nn.functional as F

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
cost_val = []
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    logits = model(data)
    loss = F.cross_entropy(logits[data.train_mask],
                           data.y[data.train_mask]) + model.l2_loss()
    loss.backward()
    optimizer.step()
    tr_acc = (logits[data.train_mask].argmax(1) ==
              data.y[data.train_mask]).float().mean().item()

    model.eval()
    with torch.no_grad():
        out = model(data)
        val_loss = (F.cross_entropy(out[data.val_mask],
                                    data.y[data.val_mask]) + model.l2_loss())
        va_acc = (out[data.val_mask].argmax(1) ==
                  data.y[data.val_mask]).float().mean().item()

    history['train_loss'].append(loss.item())
    history['val_loss'].append(val_loss.item())
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    cost_val.append(val_loss.item())

    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        print(f'epoch {epoch + 1:4d}  train_loss {loss.item():.4f}  '
              f'train_acc {tr_acc:.4f}  val_loss {val_loss.item():.4f}  '
              f'val_acc {va_acc:.4f}')

    if epoch > PATIENCE and cost_val[-1] > np.mean(cost_val[-(PATIENCE + 1):-1]):
        print(f'\nEarly stopping at epoch {epoch + 1}')
        break

print(f'\n训练结束，共 {len(cost_val)} 轮，耗时 {time.time() - t0:.2f}s')

epoch    1  train_loss 1.9462  train_acc 0.1143  val_loss 1.9415  val_acc 0.5040
epoch   11  train_loss 1.8290  train_acc 0.8143  val_loss 1.8612  val_acc 0.6040
epoch   21  train_loss 1.6432  train_acc 0.8500  val_loss 1.7402  val_acc 0.6100
epoch   31  train_loss 1.3990  train_acc 0.9000  val_loss 1.5840  val_acc 0.6640
epoch   41  train_loss 1.1192  train_acc 0.9000  val_loss 1.4051  val_acc 0.7160
epoch   51  train_loss 0.8092  train_acc 0.9571  val_loss 1.2281  val_acc 0.7540
epoch   61  train_loss 0.6228  train_acc 0.9500  val_loss 1.0743  val_acc 0.7660
epoch   71  train_loss 0.4596  train_acc 0.9714  val_loss 0.9538  val_acc 0.7760
epoch   81  train_loss 0.3440  train_acc 0.9714  val_loss 0.8687  val_acc 0.7800
epoch   91  train_loss 0.2543  train_acc 0.9857  val_loss 0.8138  val_acc 0.7860
epoch  101  train_loss 0.1825  train_acc 0.9857  val_loss 0.7744  val_acc 0.7780
epoch  111  train_loss 0.1737  train_acc 0.9714  val_loss 0.7483  val_acc 0.7820
epoch  121  train_loss 0.121

In [5]:
display(svg_lines({'train_loss': history['train_loss'],
                   'val_loss': history['val_loss']},
                  title='损失曲线', y_label='loss'))
display(svg_lines({'train_acc': history['train_acc'],
                   'val_acc': history['val_acc']},
                  title='准确率曲线', y_label='acc'))

## 4 测试与分类别准确率

In [6]:
model.eval()
with torch.no_grad():
    out = model(data)
    test_loss = (F.cross_entropy(out[data.test_mask],
                                 data.y[data.test_mask]) + model.l2_loss()).item()
    pred = out[data.test_mask].argmax(1)
    gold = data.y[data.test_mask]

acc = (pred == gold).float().mean().item()
print(f'测试集 loss  : {test_loss:.5f}')
print(f'测试集准确率 : {acc:.4f}  （{int((pred == gold).sum())}/{len(gold)}）')

rows = []
for c in range(data.num_classes):
    m = gold == c
    if int(m.sum()) == 0:
        continue
    rows.append([c, int(m.sum()), int((pred[m] == c).sum()),
                 f'{100 * (pred[m] == c).float().mean():.1f}%'])
display(html_table(['类别', '样本数', '预测正确', '该类准确率'], rows,
                   caption='分类别准确率（测试集，Cora）'))

测试集 loss  : 0.66514
测试集准确率 : 0.7950  （795/1000）


类别,样本数,预测正确,该类准确率
0,130,103,79.2%
1,91,76,83.5%
2,144,128,88.9%
3,319,238,74.6%
4,149,124,83.2%
5,103,75,72.8%
6,64,51,79.7%


## 5 层数实验：过平滑的直接证据

这是 PyG 版独有的能力——手写实现固定两层，这里可以一键加深。

两个现象值得注意：

1. **退化不是到 8 层才发生**——4 层已经跌到 75% 左右，低于 2 层的 79.5%。
   这正是论文「GCN 通常只堆两层」的经验来源。
2. **8 层只剩 6% 左右，比 7 分类的随机水平（14.3%）还差**。这不是「学不动」，
   而是**过平滑**（见 `notes/06-深度困境-过平滑与过挤压.md`）：反复做邻域平均后所有节点的
   表示收敛到同一点，分类器对任何输入都给出同一个答案。

In [7]:
rows, depth_acc = [], []
for L in (1, 2, 4, 8):
    np.random.seed(SEED); torch.manual_seed(SEED)
    m = PyGGCN(input_dim=data.x.shape[1], output_dim=data.num_classes,
               hidden1=HIDDEN, num_layers=L, dropout=DROPOUT, weight_decay=WD)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    # 与 train_pyg.py 逐行相同的协议：验证损失 = CE + L2，早停看总损失均值，
    # 且**不做权重回滚**（用最后一轮的权重评估）。协议不一致会带来 0.1~0.2
    # 个百分点的假差异——这里刻意照抄，保证 2 层能复现 79.50%。
    cv = []
    for ep in range(EPOCHS):
        m.train(); opt.zero_grad()
        lg = m(data)
        (F.cross_entropy(lg[data.train_mask], data.y[data.train_mask])
         + m.l2_loss()).backward()
        opt.step()
        m.eval()
        with torch.no_grad():
            vl = (F.cross_entropy(m(data)[data.val_mask],
                                  data.y[data.val_mask]) + m.l2_loss()).item()
        cv.append(vl)
        if ep > PATIENCE and cv[-1] > np.mean(cv[-(PATIENCE + 1):-1]):
            break
    m.eval()
    with torch.no_grad():
        a = (m(data)[data.test_mask].argmax(1) ==
             data.y[data.test_mask]).float().mean().item()
    depth_acc.append(a)
    rows.append([L, f'{count_parameters(m):,}', f'{a * 100:.2f}%',
                 '过平滑，低于随机水平' if L >= 8 else ''])

display(html_table(['层数', '参数量', '测试准确率', '备注'], rows,
                   caption='Cora 上不同深度的表现（200 轮上限 + 早停，与 train_pyg.py 一致）'))
print(f'7 分类的随机水平 = 1/7 = {100 / data.num_classes:.1f}%')

层数,参数量,测试准确率,备注
1,"10,031",73.10%,
2,"23,040",79.50%,
4,"23,552",75.40%,
8,"24,576",6.40%,过平滑，低于随机水平


7 分类的随机水平 = 1/7 = 14.3%


## 6 与手写层的数值等价性

把权重转置对齐后（手写存 `(in,out)`、PyG 存 `(out,in)`），
两者前向输出应当逐元素接近。这一项能证明 PyG 版不是「另一个模型」。

In [8]:
from gcn.layers import GraphConvolution

with silence_sparse_warning():
    _, feat, _, _, _, _, _, _, araw = load_data(DATASET)

torch.manual_seed(0)
ours = GraphConvolution(data.x.shape[1], 16, [preprocess_adj(araw)],
                        dropout=0.0, act=lambda v: v)
pyg = GCNConv(data.x.shape[1], 16, bias=False,
              add_self_loops=True, normalize=True)
with torch.no_grad():
    pyg.lin.weight.copy_(ours.weights[0].t())

with torch.no_grad():
    a = ours(feat)                       # 手写层：吃稀疏输入
    b = pyg(data.x, data.edge_index)     # PyG 层：吃稠密输入

diff = (a - b).abs().max().item()
print(f'前向输出最大偏差: {diff:.3e}')
print('结论: 两者数学定义一致，偏差仅来自浮点累加顺序' if diff < 1e-5
      else '⚠️ 偏差偏大，检查输入是否来自同一数据源')

前向输出最大偏差: 5.588e-09
结论: 两者数学定义一致，偏差仅来自浮点累加顺序


## 7 已知差异：自环的两种约定

两套实现在**原始数据自带自环**的图上并不等价：

| | 做法 | 对角元 |
|---|---|---|
| 手写 `preprocess_adj`（TF 原版） | `Ã = A + I` | 原为 1 则变成 **2** |
| PyG `gcn_norm` | `add_self_loops` 覆盖 | 恒为 **1** |

下面直接量出差异。Cora 没有自环，两者一致到 `1e-8`；Citeseer 有 124 个自环，
差异会明显变大——这也是 Citeseer 上 PyG 比手写版高 2.7 个百分点的主因之一。

In [9]:
import scipy.sparse as sp
from torch_geometric.nn.conv.gcn_conv import gcn_norm

rows = []
for ds in ('cora', 'citeseer', 'pubmed'):
    with silence_sparse_warning():
        d = to_pyg_data(ds)
        araw_ds = load_data(ds)[8]
    ours_n = preprocess_adj(araw_ds).to_dense()
    ei, ew = gcn_norm(d.edge_index, None, num_nodes=d.num_nodes,
                      add_self_loops=True, dtype=torch.float32)
    pyg_n = torch.sparse_coo_tensor(torch.stack([ei[0], ei[1]]), ew,
                                    (d.num_nodes,) * 2).to_dense()
    diff = (ours_n - pyg_n).abs().max().item()
    rows.append([ds, int((sp.coo_matrix(araw_ds).row ==
                          sp.coo_matrix(araw_ds).col).sum()),
                 f'{diff:.1e}',
                 '一致' if diff < 1e-6 else '**不同**（自环被 +1 变成 2）'])

display(html_table(['数据集', '原始自环数', '归一化矩阵最大差异', '判定'], rows,
                   caption='两种自环约定造成的差异'))

数据集,原始自环数,归一化矩阵最大差异,判定
cora,0,3.0e-08,一致
citeseer,124,1.7e-01,**不同**（自环被 +1 变成 2）
pubmed,3,9.7e-02,**不同**（自环被 +1 变成 2）


## 8 小结

| 项目 | 手写版 | PyG 版 |
|---|---:|---:|
| Cora | 76.70% | **79.50%** |
| Citeseer | 64.10% | **66.80%** |
| Pubmed | 75.60% | **76.70%** |
| 层数 | 固定 2 | 任意 |

**该用哪个**：

- 要**说明移植保真度** → 手写版（配 `test.py` 的 54 项断言，证明数学正确）
- 要**与论文可比** → PyG 版（更接近 81.5%，社区标准实现）
- 要**做过平滑/加深实验** → 只能 PyG 版

> `--num_layers 8` 在 Cora 上掉到 6.40%，这个数字比任何文字都更能说明
> 「为什么 GCN 通常只堆两层」。